# Stage 5 — Final Model: Two-Level Stacking Ensemble (`StackingClassifier`)
**Project:** 10-Year Coronary Heart Disease (CHD) Risk Prediction (Framingham Heart Study)  
**Author:** Saif Asaad Naeem | Al-Maaqal University — Department of AI Engineering

---
### Notebook Roadmap:
1. **Environment Setup & Data Loading:** Load `framingham_balanced.csv` with the exact 80%/20% stratified split (`Train = 4,917`, `Test = 1,230`).
2. **Configure Level-0 Base Learners (Top 4 Models):** Assemble the top four tuned classifiers (`1. SVC`, `2. KNN`, `3. ExtraTrees`, `4. HistGradientBoosting`), excluding 5th-place `RandomForest`.
3. **Configure & Tune Level-1 Meta-Model (`SVC`):** Set the #1 ranked model (`SVC`) as the `final_estimator` and optimize its meta-parameters via `RandomizedSearchCV ()` followed by `GridSearchCV []` (with `passthrough=True`).
4. **Master Leaderboard Compilation:** Compare all five individual models against the direct and tuned Stacking Ensembles across all 5 CV and Test metrics.
5. **Publication-Grade Final Visualizations:** Generate `fig8` (Leaderboard & Multi-Model ROC Curves) and `fig9` (Final Confusion Matrix: `TP = 703`, `FN = 8`, `Recall = 98.87%`, `F1 = 98.53%`, `ROC-AUC = 99.88%`).

### 1. Environment Setup and Data Loading
Import ensemble and evaluation libraries and prepare the stratified train/test split.

In [ ]:
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300

df = pd.read_csv('framingham_balanced.csv')
X = df.drop('TenYearCHD', axis=1).values
y = df['TenYearCHD'].values

# Unified stratified 80%/20% split (identical across all notebooks, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

def evaluate_model(model, model_name):
    cv_res = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'Model': model_name,
        'CV_Recall (%)': round(cv_res['test_recall'].mean() * 100, 2),
        'CV_Precision (%)': round(cv_res['test_precision'].mean() * 100, 2),
        'CV_F1 (%)': round(cv_res['test_f1'].mean() * 100, 2),
        'CV_ROC_AUC (%)': round(cv_res['test_roc_auc'].mean() * 100, 2),
        'CV_Accuracy (%)': round(cv_res['test_accuracy'].mean() * 100, 2),
        'Test_Recall (%)': round(recall_score(y_test, y_pred) * 100, 2),
        'Test_Precision (%)': round(precision_score(y_test, y_pred) * 100, 2),
        'Test_F1 (%)': round(f1_score(y_test, y_pred) * 100, 2),
        'Test_ROC_AUC (%)': round(roc_auc_score(y_test, y_prob) * 100, 2),
        'Test_Accuracy (%)': round(accuracy_score(y_test, y_pred) * 100, 2)
    }

print(f"Dataset Loaded Successfully: Train = {X_train.shape}, Test = {X_test.shape}")

### 2. Configure Top 4 Level-0 Base Learners
Instantiate the top four tuned classifiers using their optimal `GridSearchCV []` hyperparameters (excluding Rank #5 `RandomForest`).

In [ ]:
# Rank #1 Winner: Model 5 - SVC (CV F1 = 97.09%, Test F1 = 98.25%)
best_svc = SVC(
    C=3.905, gamma=0.6376, kernel='rbf',
    probability=True, random_state=42
)

# Rank #2: Model 4 - KNN (CV F1 = 96.73%, Test F1 = 97.43%)
best_knn = KNeighborsClassifier(
    leaf_size=30, n_neighbors=2, p=1.81,
    weights='uniform', n_jobs=-1
)

# Rank #3: Model 1 - ExtraTrees (CV F1 = 93.74%, Test F1 = 95.02%)
best_et = ExtraTreesClassifier(
    max_depth=33, max_features=0.324, min_samples_leaf=1,
    min_samples_split=2, n_estimators=168, random_state=42, n_jobs=-1
)

# Rank #4: Model 3 - HistGradientBoosting (CV F1 = 93.32%, Test F1 = 94.76%)
best_histgb = HistGradientBoostingClassifier(
    l2_regularization=0.024, learning_rate=0.043, max_depth=22,
    max_iter=453, max_leaf_nodes=77, min_samples_leaf=7, random_state=42
)

# Assemble Level-0 Base Learners
top_4_estimators = [
    ('svc', best_svc),
    ('knn', best_knn),
    ('et', best_et),
    ('histgb', best_histgb)
]

print("Top 4 Base Models Configured: [1. SVC, 2. KNN, 3. ExtraTrees, 4. HistGradientBoosting]")
print("Meta-Model (Final Estimator): SVC (Rank #1 Winner)")

### 3. Build Direct Stacking & Two-Stage Tuning `() -> []` of Level-1 Meta-Model (`SVC`)
Evaluate direct Stacking first, then optimize the Level-1 `SVC` Meta-Model via `RandomizedSearchCV ()` followed by `GridSearchCV []`.

In [ ]:
# 1. Direct Stacking using the top-ranked SVC as the Level-1 Meta-Model
stack_direct = StackingClassifier(
    estimators=top_4_estimators,
    final_estimator=SVC(C=3.905, gamma=0.6376, kernel='rbf', probability=True, random_state=42),
    cv=skf,
    stack_method='predict_proba',
    passthrough=False,
    n_jobs=-1
)
res_stack_direct = evaluate_model(stack_direct, '1. Stacking (Direct Best SVC Meta-Model)')
print("Direct Stacking Evaluation:", res_stack_direct)

# 2. Stage 1 Meta-Model Tuning: Continuous Exploration via RandomizedSearchCV ()
param_dist_meta = {
    'final_estimator__C': loguniform(0.1, 20.0),
    'final_estimator__gamma': loguniform(0.01, 5.0)
}

rand_stack = RandomizedSearchCV(
    estimator=StackingClassifier(
        estimators=top_4_estimators,
        final_estimator=SVC(kernel='rbf', probability=True, random_state=42),
        cv=skf,
        stack_method='predict_proba',
        n_jobs=-1
    ),
    param_distributions=param_dist_meta,
    n_iter=15,
    scoring='f1',
    cv=skf,
    random_state=42,
    n_jobs=-1
)
rand_stack.fit(X_train, y_train)
best_meta_r = rand_stack.best_params_
print("Best Meta-Model Params from RandomizedSearchCV ():", best_meta_r)
res_stack_rand = evaluate_model(rand_stack.best_estimator_, '2. Stacking (RandomizedSearchCV Meta-SVC)')

# 3. Stage 2 Meta-Model Tuning: Exhaustive Grid Exploitation via GridSearchCV []
c_val = best_meta_r['final_estimator__C']
g_val = best_meta_r['final_estimator__gamma']

param_grid_meta = {
    'final_estimator__C': [round(c_val * 0.7, 3), round(c_val, 3), round(c_val * 1.3, 3), 1.0, 3.905],
    'final_estimator__gamma': [round(g_val * 0.7, 4), round(g_val, 4), round(g_val * 1.3, 4), 0.6376],
    'passthrough': [False, True]
}

grid_stack = GridSearchCV(
    estimator=StackingClassifier(
        estimators=top_4_estimators,
        final_estimator=SVC(kernel='rbf', probability=True, random_state=42),
        cv=skf,
        stack_method='predict_proba',
        n_jobs=-1
    ),
    param_grid=param_grid_meta,
    scoring='f1',
    cv=skf,
    n_jobs=-1
)
grid_stack.fit(X_train, y_train)
print("Best Meta-Model Params from GridSearchCV []:", grid_stack.best_params_)
res_stack_grid = evaluate_model(grid_stack.best_estimator_, '3. Stacking Ensemble (Final GridSearchCV [])')

# Serialize the winning Stacking Ensemble model
best_stacking_model = grid_stack.best_estimator_
joblib.dump(best_stacking_model, 'best_model6_stacking.pkl')

### 4. Final Master Leaderboard Table
Compile the comprehensive leaderboard comparing all five individual tuned models and the Stacking Ensemble.

In [ ]:
individual_summaries = [
    {
        'Model': 'Model 2: RandomForest (Rank 5 - Excluded)',
        'CV_Recall (%)': 94.30, 'CV_Precision (%)': 88.35, 'CV_F1 (%)': 91.22, 'CV_ROC_AUC (%)': 95.93, 'CV_Accuracy (%)': 89.51,
        'Test_Recall (%)': 94.37, 'Test_Precision (%)': 90.07, 'Test_F1 (%)': 92.17, 'Test_ROC_AUC (%)': 96.83, 'Test_Accuracy (%)': 90.73
    },
    {
        'Model': 'Model 3: HistGradientBoosting (Rank 4 - Base 4)',
        'CV_Recall (%)': 95.36, 'CV_Precision (%)': 91.38, 'CV_F1 (%)': 93.32, 'CV_ROC_AUC (%)': 97.39, 'CV_Accuracy (%)': 92.11,
        'Test_Recall (%)': 95.36, 'Test_Precision (%)': 94.17, 'Test_F1 (%)': 94.76, 'Test_ROC_AUC (%)': 98.37, 'Test_Accuracy (%)': 93.90
    },
    {
        'Model': 'Model 1: ExtraTrees (Rank 3 - Base 3)',
        'CV_Recall (%)': 95.92, 'CV_Precision (%)': 91.68, 'CV_F1 (%)': 93.74, 'CV_ROC_AUC (%)': 97.96, 'CV_Accuracy (%)': 92.60,
        'Test_Recall (%)': 96.62, 'Test_Precision (%)': 93.47, 'Test_F1 (%)': 95.02, 'Test_ROC_AUC (%)': 98.58, 'Test_Accuracy (%)': 94.15
    },
    {
        'Model': 'Model 4: KNN (Rank 2 - Base 2)',
        'CV_Recall (%)': 97.40, 'CV_Precision (%)': 96.08, 'CV_F1 (%)': 96.73, 'CV_ROC_AUC (%)': 96.77, 'CV_Accuracy (%)': 96.20,
        'Test_Recall (%)': 98.73, 'Test_Precision (%)': 96.16, 'Test_F1 (%)': 97.43, 'Test_ROC_AUC (%)': 97.09, 'Test_Accuracy (%)': 96.99
    },
    {
        'Model': 'Model 5: SVC (Rank 1 - Base 1 & Meta-Model)',
        'CV_Recall (%)': 96.45, 'CV_Precision (%)': 97.76, 'CV_F1 (%)': 97.09, 'CV_ROC_AUC (%)': 99.50, 'CV_Accuracy (%)': 96.66,
        'Test_Recall (%)': 98.45, 'Test_Precision (%)': 98.04, 'Test_F1 (%)': 98.25, 'Test_ROC_AUC (%)': 99.78, 'Test_Accuracy (%)': 97.97
    },
    res_stack_direct,
    res_stack_grid
]

final_leaderboard_df = pd.DataFrame(individual_summaries)
print("\n=== Comprehensive Final Leaderboard (All Individual Models vs. Stacking Ensemble) ===")
print(final_leaderboard_df.to_string(index=False))

### 5. Final Presentation Visualizations (Figures 8 & 9)
Generate the multi-model metric comparison, overlaid ROC curves, final Stacking Confusion Matrix, and Precision-Recall curve.

In [ ]:
# Figure 8: Final Models vs. Stacking Ensemble Leaderboard & Overlaid ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

plot_df = final_leaderboard_df.iloc[[0, 1, 2, 3, 4, 6]].copy()
short_labels = ['RandomForest\n(Rank 5)', 'HistGB\n(Base 4)', 'ExtraTrees\n(Base 3)', 'KNN\n(Base 2)', 'SVC\n(Base 1)', 'STACKING\n(Final Winner)']
x_pos = np.arange(len(short_labels))
w = 0.20

axes[0].bar(x_pos - 1.5*w, plot_df['Test_Recall (%)'], width=w, label='Recall (%)', color='#e11d48', edgecolor='black')
axes[0].bar(x_pos - 0.5*w, plot_df['Test_Precision (%)'], width=w, label='Precision (%)', color='#8b5cf6', edgecolor='black')
axes[0].bar(x_pos + 0.5*w, plot_df['Test_F1 (%)'], width=w, label='F1-Score (%)', color='#f59e0b', edgecolor='black')
axes[0].bar(x_pos + 1.5*w, plot_df['Test_ROC_AUC (%)'], width=w, label='ROC-AUC (%)', color='#10b981', edgecolor='black')

for i, f1_v in enumerate(plot_df['Test_F1 (%)']):
    axes[0].text(x_pos[i] + 0.5*w, f1_v + 0.4, f"{f1_v:.2f}%", ha='center', fontsize=8, fontweight='bold')

axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(short_labels, fontweight='bold', fontsize=9)
axes[0].set_ylim(85, 101.5)
axes[0].set_ylabel('Test Score (%)')
axes[0].set_title('A. Final Models vs. Stacking Ensemble (Recall, Precision, F1, ROC-AUC)', fontweight='bold')
axes[0].legend(loc='lower right', ncol=2, frameon=True)

for name_m, est_m in top_4_estimators:
    est_m.fit(X_train, y_train)
    RocCurveDisplay.from_estimator(est_m, X_test, y_test, ax=axes[1], name=name_m.upper(), alpha=0.75)

RocCurveDisplay.from_estimator(
    best_stacking_model, X_test, y_test, ax=axes[1],
    name='STACKING ENSEMBLE (Meta=SVC)', color='black', linewidth=2.5
)
axes[1].set_title('B. ROC Curves Comparison on Test Set', fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_final_models_and_stacking_leaderboard.png', bbox_inches='tight')
plt.show()

# Figure 9: Final Stacking Ensemble Confusion Matrix & Precision-Recall Curve
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
y_pred_stack = best_stacking_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_stack)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', cbar=False, annot_kws={'size': 13, 'weight': 'bold'}, ax=axes[0])
axes[0].set_title('A. Final Stacking Ensemble Confusion Matrix (Test Set)', fontweight='bold')
axes[0].set_xlabel('Predicted Label (0 = Healthy, 1 = CHD)', fontweight='bold')
axes[0].set_ylabel('True Label (0 = Healthy, 1 = CHD)', fontweight='bold')

PrecisionRecallDisplay.from_estimator(
    best_stacking_model, X_test, y_test, ax=axes[1],
    name='Stacking Ensemble', color='#e11d48', linewidth=2.2
)
axes[1].set_title('B. Precision-Recall Curve (Stacking Ensemble)', fontweight='bold')
plt.tight_layout()
plt.savefig('fig9_stacking_confusion_matrix_and_pr_curve.png', bbox_inches='tight')
plt.show()